In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import TransformerConv, global_add_pool
from pymatgen.core import Structure
from scipy.spatial.distance import cdist
from matbench.bench import MatbenchBenchmark
from sklearn.metrics import mean_absolute_error
import math
np.math = math

# Load Matbench dataset
mb = MatbenchBenchmark(autoload=False)
task = mb.matbench_phonons
task.load()

print(f"✅ Dataset: {len(task.df)} samples")
print(f"   Target range: [{task.df['last phdos peak'].min():.1f}, {task.df['last phdos peak'].max():.1f}] cm⁻¹")

c:\Users\Victor\.conda\envs\pytorch-matbench\lib\site-packages\torch_geometric\typing.py:86: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: [WinError 127] The specified procedure could not be found
  warnings.warn(f"An issue occurred while importing 'torch-scatter'. "
c:\Users\Victor\.conda\envs\pytorch-matbench\lib\site-packages\torch_geometric\typing.py:97: UserWarning: An issue occurred while importing 'torch-cluster'. Disabling its usage. Stacktrace: [WinError 127] The specified procedure could not be found
  warnings.warn(f"An issue occurred while importing 'torch-cluster'. "
c:\Users\Victor\.conda\envs\pytorch-matbench\lib\site-packages\torch_geometric\typing.py:113: UserWarning: An issue occurred while importing 'torch-spline-conv'. Disabling its usage. Stacktrace: [WinError 127] The specified procedure could not be found
  warnings.warn(
c:\Users\Victor\.conda\envs\pytorch-matbench\lib\site-packages\torch_geometric\typing.py:124

2025-10-21 20:59:06 INFO     Initialized benchmark 'matbench_v0.1' with 13 tasks: 
['matbench_dielectric',
 'matbench_expt_gap',
 'matbench_expt_is_metal',
 'matbench_glass',
 'matbench_jdft2d',
 'matbench_log_gvrh',
 'matbench_log_kvrh',
 'matbench_mp_e_form',
 'matbench_mp_gap',
 'matbench_mp_is_metal',
 'matbench_perovskites',
 'matbench_phonons',
 'matbench_steels']
2025-10-21 20:59:06 INFO     Loading dataset 'matbench_phonons'...
2025-10-21 20:59:06 INFO     Dataset 'matbench_phonons loaded.
✅ Dataset: 1265 samples
   Target range: [59.6, 3643.7] cm⁻¹


In [2]:
# Load periodic table (76 features per element)
df_ptable = pd.read_csv('ptable.csv')
df_ptable.fillna(0, inplace=True)
df_ptable.drop(['electronic_configuration', 'name', 'block', 'lattice_structure', 'is_radioactive'], 
               axis=1, inplace=True, errors='ignore')

n_features = len(df_ptable.columns) - 1  # Exclude 'symbol' column
print(f"✅ Loaded ptable: {n_features} features/atom")

✅ Loaded ptable: 76 features/atom


In [3]:
def structure_to_graph(structure, target_value, df_ptable):
    """Convert crystal structure to bond graph: nodes=bonds, edges=angles"""
    structure = structure.copy()
    structure.make_supercell([1, 1, 1])
    
    positions = np.array([site.coords for site in structure.sites])
    distances = cdist(positions, positions)
    cutoff = 8
    
    # پیدا کردن تمام پیوندها
    bonds = []
    for i in range(len(positions)):
        for j in range(i + 1, len(positions)):
            if distances[i, j] <= cutoff:
                bonds.append((i, j))
    
    if len(bonds) == 0:
        # اگر هیچ پیوندی نبود، یک نود خالی بساز
        x = torch.zeros((1, 81), dtype=torch.float)
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr = torch.zeros((0, 1), dtype=torch.float)
        y = torch.tensor([target_value], dtype=torch.float)
        
        # اضافه کردن lattice info
        lattice = structure.lattice
        spacegroup_num = 1  # default
        try:
            from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
            sga = SpacegroupAnalyzer(structure)
            spacegroup_num = sga.get_space_group_number()
        except:
            pass
        
        u = torch.tensor([[
            structure.volume / len(positions),
            spacegroup_num,
            lattice.volume
        ]], dtype=torch.float)
        return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y, u=u)
    
    # ساخت node features برای هر پیوند
    node_features = []
    for (i, j) in bonds:
        Z_i = structure.sites[i].specie.Z
        Z_j = structure.sites[j].specie.Z
        
        features_i = df_ptable.iloc[Z_i - 1, 1:].values.astype(np.float32)
        features_j = df_ptable.iloc[Z_j - 1, 1:].values.astype(np.float32)
        
        bond_feature = np.zeros(1)
        
        # اضافه کردن فاصله پیوند و coordination number
        bond_distance = distances[i, j]
        coord_i = np.sum(distances[i, :] <= cutoff) - 1
        coord_j = np.sum(distances[j, :] <= cutoff) - 1
        
        # اضافه کردن lattice info
        lattice = structure.lattice
        spacegroup_num = 1  # default
        lattice_type = 0  # cubic=0, tetragonal=1, orthorhombic=2, etc.
        
        try:
            from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
            sga = SpacegroupAnalyzer(structure)
            spacegroup_num = sga.get_space_group_number()
            crystal_system = sga.get_crystal_system()
            
            # نوع شبکه رو encode کن
            system_map = {
                'cubic': 0, 'tetragonal': 1, 'orthorhombic': 2,
                'hexagonal': 3, 'trigonal': 4, 'monoclinic': 5, 'triclinic': 6
            }
            lattice_type = system_map.get(crystal_system, 0)
        except:
            pass
        
        bond_feature = np.concatenate([
            bond_feature, 
            [bond_distance, (coord_i + coord_j) / 2, spacegroup_num / 230.0, lattice_type / 6.0, lattice.volume]
        ])
        node_features.append(bond_feature)
    
    x = torch.tensor(node_features, dtype=torch.float)
    
    # ساخت edges بین پیوندها (زوایا)
    edge_index = []
    edge_attr = []
    
    for bond_idx1, (i1, j1) in enumerate(bonds):
        for bond_idx2, (i2, j2) in enumerate(bonds):
            if bond_idx1 != bond_idx2:
                # اتم مشترک پیدا کن
                shared_atom = None
                if i1 in (i2, j2):
                    shared_atom = i1
                    other1, other2 = j1, (j2 if i1 == i2 else i2)
                elif j1 in (i2, j2):
                    shared_atom = j1
                    other1, other2 = i1, (j2 if j1 == i2 else i2)
                
                if shared_atom is not None:
                    # محاسبه زاویه
                    vec1 = positions[other1] - positions[shared_atom]
                    vec2 = positions[other2] - positions[shared_atom]
                    
                    cos_angle = np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))
                    cos_angle = np.clip(cos_angle, -1, 1)
                    angle = np.arccos(cos_angle)
                    
                    edge_index.append([bond_idx1, bond_idx2])
                    edge_attr.append([angle])
    
    if len(edge_index) == 0:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr = torch.zeros((0, 1), dtype=torch.float)
    else:
        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
        edge_attr = torch.tensor(edge_attr, dtype=torch.float)
    
    y = torch.tensor([target_value], dtype=torch.float)
    
    # Global features با lattice info
    lattice = structure.lattice
    spacegroup_num = 1
    try:
        from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
        sga = SpacegroupAnalyzer(structure)
        spacegroup_num = sga.get_space_group_number()
    except:
        pass
    
    u = torch.tensor([[
        structure.volume / len(positions),
        spacegroup_num,
        lattice.volume
    ]], dtype=torch.float)
    
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y, u=u)


In [4]:
def structure_to_graph2(structure, target_value, df_ptable):
    """Convert crystal structure to fully connected graph with 76 features/node"""
    # ساخت supercell 3x3x3
    structure = structure.copy()
    structure.make_supercell([3, 3, 3])
    
    # Extract node features (76 features per atom)
    node_features = []
    atomic_numbers = []
    
    for site in structure.sites:
        Z = site.specie.Z
        atomic_numbers.append(Z)
        # Get features using atomic number as index (Z=1 → row 0)
        features = df_ptable.iloc[Z - 1, 1:].values.astype(np.float32)
        node_features.append(features)
    
    x = torch.tensor(node_features, dtype=torch.float)
    z = torch.tensor(atomic_numbers, dtype=torch.long)
    
    # Positions
    positions = np.array([site.coords for site in structure.sites])
    pos = torch.tensor(positions, dtype=torch.float)
    
    # Edges با cutoff 5 انگستروم در کل supercell
    distances = cdist(positions, positions)
    edge_index = []
    edge_attr = []
    
    cutoff = 8  # انگستروم
    
    n_atoms = len(positions)
    for i in range(n_atoms):
        for j in range(n_atoms):
            if i != j and distances[i, j] <= cutoff:
                edge_index.append([i, j])
                edge_attr.append([distances[i, j]])
    
    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(edge_attr, dtype=torch.float)
    
    # Target and global features
    y = torch.tensor([target_value], dtype=torch.float)
    u = torch.tensor([[structure.volume / n_atoms]], dtype=torch.float)
    
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y, pos=pos, z=z, u=u)

In [9]:
def structure_to_graph3(structure, target_value, df_ptable):
    """Convert crystal structure to torsion graph: nodes=planes(3-body), edges=torsions(4-body)"""
    structure = structure.copy()
    structure.make_supercell([1, 1, 1])
    
    positions = np.array([site.coords for site in structure.sites])
    distances = cdist(positions, positions)
    cutoff = 8
    
    # پیدا کردن تمام پیوندها
    bonds = []
    for i in range(len(positions)):
        for j in range(i + 1, len(positions)):
            if distances[i, j] <= cutoff:
                bonds.append((i, j))
    
    if len(bonds) == 0:
        x = torch.zeros((1, 6), dtype=torch.float)
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr = torch.zeros((0, 1), dtype=torch.float)
        y = torch.tensor([target_value], dtype=torch.float)
        
        lattice = structure.lattice
        spacegroup_num = 1
        try:
            from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
            sga = SpacegroupAnalyzer(structure)
            spacegroup_num = sga.get_space_group_number()
        except:
            pass
        
        u = torch.tensor([[
            structure.volume / len(positions),
            spacegroup_num,
            lattice.volume
        ]], dtype=torch.float)
        return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y, u=u)
    
    # پیدا کردن زوایا (3-body) از گراف 1 - هر یال گراف 1 = یک نود در گراف 3
    angles = []  # هر angle = (bond_idx1, bond_idx2, shared_atom, angle_value)
    
    for bond_idx1, (i1, j1) in enumerate(bonds):
        for bond_idx2, (i2, j2) in enumerate(bonds):
            if bond_idx1 < bond_idx2:
                shared_atom = None
                if i1 in (i2, j2):
                    shared_atom = i1
                    other1, other2 = j1, (j2 if i1 == i2 else i2)
                elif j1 in (i2, j2):
                    shared_atom = j1
                    other1, other2 = i1, (j2 if j1 == i2 else i2)
                
                if shared_atom is not None:
                    vec1 = positions[other1] - positions[shared_atom]
                    vec2 = positions[other2] - positions[shared_atom]
                    
                    cos_angle = np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))
                    cos_angle = np.clip(cos_angle, -1, 1)
                    angle = np.arccos(cos_angle)
                    
                    angles.append((bond_idx1, bond_idx2, shared_atom, angle))
    
    if len(angles) == 0:
        x = torch.zeros((1, 6), dtype=torch.float)
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr = torch.zeros((0, 1), dtype=torch.float)
        y = torch.tensor([target_value], dtype=torch.float)
        
        lattice = structure.lattice
        spacegroup_num = 1
        try:
            from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
            sga = SpacegroupAnalyzer(structure)
            spacegroup_num = sga.get_space_group_number()
        except:
            pass
        
        u = torch.tensor([[
            structure.volume / len(positions),
            spacegroup_num,
            lattice.volume
        ]], dtype=torch.float)
        return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y, u=u)
    
    # ساخت node features برای هر صفحه (3-body)
    node_features = []
    for (bond_idx1, bond_idx2, shared_atom, angle_value) in angles:
        i1, j1 = bonds[bond_idx1]
        i2, j2 = bonds[bond_idx2]
        
        bond_dist1 = distances[i1, j1]
        bond_dist2 = distances[i2, j2]
        
        plane_feature = np.array([
            bond_dist1,
            bond_dist2,
            angle_value,
            structure.sites[shared_atom].specie.Z / 118.0,
            (bond_dist1 + bond_dist2) / 2,
            np.sin(angle_value)
        ], dtype=np.float32)
        
        node_features.append(plane_feature)
    
    x = torch.tensor(node_features, dtype=torch.float)
    
    # ساخت edges بین صفحه‌ها (torsion angles = 4-body)
    edge_index = []
    edge_attr = []
    
    for plane_idx1, (b1_1, b1_2, shared1, angle1) in enumerate(angles):
        for plane_idx2, (b2_1, b2_2, shared2, angle2) in enumerate(angles):
            if plane_idx1 < plane_idx2:
                # پیدا کردن پیوند مشترک بین دو صفحه
                bonds_plane1 = {b1_1, b1_2}
                bonds_plane2 = {b2_1, b2_2}
                shared_bond = bonds_plane1 & bonds_plane2
                
                if len(shared_bond) == 1:
                    shared_bond_idx = list(shared_bond)[0]
                    atom_i, atom_j = bonds[shared_bond_idx]
                    
                    # پیوند اول صفحه 1
                    other_bond1 = (b1_1 if b1_2 == shared_bond_idx else b1_2)
                    atom1 = None
                    if bonds[other_bond1][0] not in (atom_i, atom_j):
                        atom1 = bonds[other_bond1][0]
                    elif bonds[other_bond1][1] not in (atom_i, atom_j):
                        atom1 = bonds[other_bond1][1]
                    
                    # پیوند دوم صفحه 2
                    other_bond2 = (b2_1 if b2_2 == shared_bond_idx else b2_2)
                    atom4 = None
                    if bonds[other_bond2][0] not in (atom_i, atom_j):
                        atom4 = bonds[other_bond2][0]
                    elif bonds[other_bond2][1] not in (atom_i, atom_j):
                        atom4 = bonds[other_bond2][1]
                    
                    if atom1 is not None and atom4 is not None:
                        # محاسبه torsion angle
                        v1 = positions[atom1] - positions[atom_i]
                        v2 = positions[atom_j] - positions[atom_i]
                        v3 = positions[atom4] - positions[atom_j]
                        
                        n1 = np.cross(v1, v2)
                        n2 = np.cross(v2, v3)
                        
                        norm1 = np.linalg.norm(n1)
                        norm2 = np.linalg.norm(n2)
                        
                        if norm1 > 1e-6 and norm2 > 1e-6:
                            n1 = n1 / norm1
                            n2 = n2 / norm2
                            
                            cos_torsion = np.dot(n1, n2)
                            cos_torsion = np.clip(cos_torsion, -1, 1)
                            torsion_angle = np.arccos(cos_torsion)
                            
                            edge_index.append([plane_idx1, plane_idx2])
                            edge_attr.append([torsion_angle])
    
    if len(edge_index) == 0:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr = torch.zeros((0, 1), dtype=torch.float)
    else:
        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
        edge_attr = torch.tensor(edge_attr, dtype=torch.float)
    
    y = torch.tensor([target_value], dtype=torch.float)
    
    lattice = structure.lattice
    spacegroup_num = 1
    try:
        from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
        sga = SpacegroupAnalyzer(structure)
        spacegroup_num = sga.get_space_group_number()
    except:
        pass
    
    u = torch.tensor([[
        structure.volume / len(positions),
        spacegroup_num,
        lattice.volume
    ]], dtype=torch.float)
    
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y, u=u)


In [15]:
# تست گراف 3 - بررسی اینکه واقعاً torsion (4-body) رو حساب میکنه

# بیا یک ساختار ساده بسازیم و بررسی کنیم
test_structure = train_inputs.iloc[1]
test_target = train_outputs.iloc[1]

# ساخت گراف 3
graph3 = structure_to_graph3(test_structure, test_target, df_ptable)

print("=" * 60)
print("📊 بررسی گراف 3 (Torsion Graph - 4-body)")
print("=" * 60)

# اطلاعات ساختار
print(f"\n🔹 ساختار: {test_structure.composition}")
print(f"   تعداد اتم‌ها: {len(test_structure)}")

# اطلاعات نودها (صفحه‌ها / زاویه‌ها)
print(f"\n🔹 نودها (Planes/Angles - هر نود = 1 زاویه 3-جسمی):")
print(f"   تعداد نودها: {graph3.x.shape[0]}")
print(f"   ویژگی‌های هر نود: {graph3.x.shape[1]}")
print(f"   هر نود = 2 پیوند + 1 اتم مشترک = یک صفحه")

if graph3.x.shape[0] > 0:
    print(f"\n   نمونه از اولین نود (صفحه):")
    print(f"   - فاصله پیوند 1: {graph3.x[0, 0]:.3f} Å")
    print(f"   - فاصله پیوند 2: {graph3.x[0, 1]:.3f} Å")
    print(f"   - زاویه بین پیوندها: {graph3.x[0, 2]:.3f} rad = {np.degrees(graph3.x[0, 2]):.1f}°")
    print(f"   - عدد اتمی مرکزی: {graph3.x[0, 3]:.3f}")
    print(f"   - میانگین فاصله‌ها: {graph3.x[0, 4]:.3f} Å")
    print(f"   - sin(زاویه): {graph3.x[0, 5]:.3f}")

# اطلاعات یال‌ها (torsion angles)
print(f"\n🔹 یال‌ها (Torsion Angles - هر یال = 1 زاویه 4-جسمی):")
print(f"   تعداد یال‌ها: {graph3.edge_index.shape[1]}")
print(f"   هر یال = 2 صفحه با پیوند مشترک")
print(f"   وزن یال = زاویه بین دو صفحه (torsion angle)")

if graph3.edge_index.shape[1] > 0:
    print(f"\n   نمونه از اولین یال:")
    edge_src = graph3.edge_index[0, 0].item()
    edge_dst = graph3.edge_index[1, 0].item()
    torsion_angle = graph3.edge_attr[0, 0].item()
    print(f"   - اتصال: نود {edge_src} → نود {edge_dst}")
    print(f"   - زاویه torsion: {torsion_angle:.3f} rad = {np.degrees(torsion_angle):.1f}°")
    print(f"   - این یعنی: 4 اتم در یک ترتیب خاص که زاویه بین 2 صفحه را تعریف می‌کنند")

# آمار کلی
if graph3.edge_attr.shape[0] > 0:
    torsion_angles_deg = np.degrees(graph3.edge_attr.numpy())
    print(f"\n📈 آمار زاویه‌های Torsion:")
    print(f"   - میانگین: {torsion_angles_deg.mean():.1f}°")
    print(f"   - انحراف معیار: {torsion_angles_deg.std():.1f}°")
    print(f"   - کمینه: {torsion_angles_deg.min():.1f}°")
    print(f"   - بیشینه: {torsion_angles_deg.max():.1f}°")

print("\n" + "=" * 60)
print("✅ تأیید: گراف 3 واقعاً انرژی 4-جسمی (torsion) را مدل می‌کند!")
print("   - نودها = صفحه‌های 3-اتمی (از گراف 1)")
print("   - یال‌ها = زوایای بین صفحه‌ها (4 اتم)")
print("=" * 60)

# بررسی تفاوت با گراف‌های دیگر
graph1 = structure_to_graph(test_structure, test_target, df_ptable)
graph2 = structure_to_graph2(test_structure, test_target, df_ptable)

print(f"\n📊 مقایسه سه گراف:")
print(f"   گراف 1 (Bond): {graph1.x.shape[0]} نود (پیوندها), {graph1.edge_index.shape[1]} یال (زاویه‌ها 3-body)")
print(f"   گراف 2 (Atom): {graph2.x.shape[0]} نود (اتم‌ها), {graph2.edge_index.shape[1]} یال (فاصله‌ها 2-body)")
print(f"   گراف 3 (Torsion): {graph3.x.shape[0]} نود (صفحه‌ها), {graph3.edge_index.shape[1]} یال (torsion 4-body)")


📊 بررسی گراف 3 (Torsion Graph - 4-body)

🔹 ساختار: Al1 Ga1 N2
   تعداد اتم‌ها: 4

🔹 نودها (Planes/Angles - هر نود = 1 زاویه 3-جسمی):
   تعداد نودها: 12
   ویژگی‌های هر نود: 6
   هر نود = 2 پیوند + 1 اتم مشترک = یک صفحه

   نمونه از اولین نود (صفحه):
   - فاصله پیوند 1: 3.162 Å
   - فاصله پیوند 2: 1.908 Å
   - زاویه بین پیوندها: 0.627 rad = 35.9°
   - عدد اتمی مرکزی: 0.110
   - میانگین فاصله‌ها: 2.535 Å
   - sin(زاویه): 0.587

🔹 یال‌ها (Torsion Angles - هر یال = 1 زاویه 4-جسمی):
   تعداد یال‌ها: 36
   هر یال = 2 صفحه با پیوند مشترک
   وزن یال = زاویه بین دو صفحه (torsion angle)

   نمونه از اولین یال:
   - اتصال: نود 0 → نود 1
   - زاویه torsion: 0.976 rad = 55.9°
   - این یعنی: 4 اتم در یک ترتیب خاص که زاویه بین 2 صفحه را تعریف می‌کنند

📈 آمار زاویه‌های Torsion:
   - میانگین: 135.0°
   - انحراف معیار: 46.5°
   - کمینه: 55.3°
   - بیشینه: 180.0°

✅ تأیید: گراف 3 واقعاً انرژی 4-جسمی (torsion) را مدل می‌کند!
   - نودها = صفحه‌های 3-اتمی (از گراف 1)
   - یال‌ها = زوایای بین صفحه‌ها (4 اتم)

In [10]:
class CustomMessagePassing(nn.Module):
    """
    لایه Message Passing سفارشی با Attention:
    پیام = attention_weight * (ویژگی_نود_همسایه * ویژگی_یال)
    """
    def __init__(self, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        
        # Attention mechanism
        # محاسبه attention score از روی: نود مقصد + نود همسایه + یال
        self.attention_mlp = nn.Sequential(
            nn.Linear(3 * hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.SiLU(),
            nn.Linear(hidden_dim // 2, 1),
            nn.LeakyReLU(0.2)
        )
        
        # MLP برای ترکیب پیام‌ها
        self.message_mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
        )
        
    def forward(self, x, edge_index, edge_attr):
        # edge_index: [2, num_edges]
        # edge_index[0]: source nodes (همسایه‌ها)
        # edge_index[1]: target nodes (نودهای مقصد)
        
        source_nodes = edge_index[0]
        target_nodes = edge_index[1]
        
        # ویژگی نودهای همسایه و مقصد
        neighbor_features = x[source_nodes]  # [num_edges, hidden_dim]
        target_features = x[target_nodes]    # [num_edges, hidden_dim]
        
        # ============ محاسبه Attention Scores ============
        # ترکیب: نود مقصد + نود همسایه + یال
        attention_input = torch.cat([target_features, neighbor_features, edge_attr], dim=1)
        attention_scores = self.attention_mlp(attention_input)  # [num_edges, 1]
        
        # ============ Softmax سریع با scatter_softmax ============
        from torch_geometric.utils import softmax
        attention_weights = softmax(attention_scores, target_nodes, num_nodes=x.size(0))
        
        # ============ ضرب ویژگی همسایه در ویژگی یال ============
        messages = neighbor_features * edge_attr  # [num_edges, hidden_dim]
        
        # ============ اعمال Attention Weights ============
        weighted_messages = messages * attention_weights  # [num_edges, hidden_dim]
        
        # ============ Aggregate Messages ============
        num_nodes = x.size(0)
        aggregated = torch.zeros(num_nodes, self.hidden_dim, device=x.device)
        aggregated.index_add_(0, target_nodes, weighted_messages)
        
        # اعمال MLP روی پیام‌های جمع شده
        output = self.message_mlp(aggregated)
        
        return output

class TripleGraphGNN(nn.Module):
    """
    مدل سه گرافی: گراف پیوندی + گراف اتمی + گراف torsion
    """
    def __init__(self, n_bond_features=6, n_atom_features=76, n_torsion_features=6, edge_dim=1):
        super().__init__()
        hidden = 32
        
        # ============ Bond Graph Branch (3-body) ============
        self.bond_embedding = nn.Sequential(
            nn.Linear(n_bond_features, hidden),
            nn.BatchNorm1d(hidden),
            nn.SiLU(),
            nn.Dropout(0.1),
        )
        
        self.bond_edge_embedding = nn.Sequential(
            nn.Linear(edge_dim, hidden),
            nn.SiLU(),
        )
        
        self.bond_message_layers = nn.ModuleList([
            CustomMessagePassing(hidden)
            for _ in range(2)
        ])
        
        self.bond_layer_norms = nn.ModuleList([
            nn.LayerNorm(hidden)
            for _ in range(2)
        ])
        
        self.bond_attention = nn.Sequential(
            nn.Linear(hidden, hidden // 4),
            nn.SiLU(),
            nn.Linear(hidden // 4, 1),
            nn.Sigmoid()
        )
        
        # ============ Atom Graph Branch (2-body) ============
        self.atom_embedding = nn.Sequential(
            nn.Linear(n_atom_features, hidden),
            nn.BatchNorm1d(hidden),
            nn.SiLU(),
            nn.Dropout(0.1),
        )
        
        self.atom_edge_embedding = nn.Sequential(
            nn.Linear(edge_dim, hidden),
            nn.SiLU(),
        )
        
        self.atom_message_layers = nn.ModuleList([
            CustomMessagePassing(hidden)
            for _ in range(2)
        ])
        
        self.atom_layer_norms = nn.ModuleList([
            nn.LayerNorm(hidden)
            for _ in range(2)
        ])
        
        self.atom_attention = nn.Sequential(
            nn.Linear(hidden, hidden // 4),
            nn.SiLU(),
            nn.Linear(hidden // 4, 1),
            nn.Sigmoid()
        )
        
        # ============ Torsion Graph Branch (4-body) ============
        self.torsion_embedding = nn.Sequential(
            nn.Linear(n_torsion_features, hidden),
            nn.BatchNorm1d(hidden),
            nn.SiLU(),
            nn.Dropout(0.1),
        )
        
        self.torsion_edge_embedding = nn.Sequential(
            nn.Linear(edge_dim, hidden),
            nn.SiLU(),
        )
        
        self.torsion_message_layers = nn.ModuleList([
            CustomMessagePassing(hidden)
            for _ in range(2)
        ])
        
        self.torsion_layer_norms = nn.ModuleList([
            nn.LayerNorm(hidden)
            for _ in range(2)
        ])
        
        self.torsion_attention = nn.Sequential(
            nn.Linear(hidden, hidden // 4),
            nn.SiLU(),
            nn.Linear(hidden // 4, 1),
            nn.Sigmoid()
        )
        
        # ============ Pooling ============
        from torch_geometric.nn import Set2Set, global_mean_pool, global_max_pool
        self.set2set_pool = Set2Set(hidden, processing_steps=1)
        self.mean_pool = global_mean_pool
        self.max_pool = global_max_pool
        
        # ============ Global Features ============
        self.global_mlp = nn.Sequential(
            nn.Linear(3, hidden // 4),
            nn.SiLU(),
        )
        
        # ============ Final Prediction ============
        # از bond graph: 2*hidden (Set2Set) + hidden (mean) + hidden (max) = 4*hidden
        # از atom graph: 2*hidden (Set2Set) + hidden (mean) + hidden (max) = 4*hidden
        # از torsion graph: 2*hidden (Set2Set) + hidden (mean) + hidden (max) = 4*hidden
        # global: hidden//4
        final_input = 12 * hidden + hidden // 4
        
        self.final_mlp = nn.Sequential(
            nn.Linear(final_input, 256),
            nn.LayerNorm(256),
            nn.SiLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.SiLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.LayerNorm(64),
            nn.SiLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )
    
    def forward(self, bond_data, atom_data, torsion_data):
        # ============ Bond Graph Forward (3-body) ============
        x_bond = self.bond_embedding(bond_data.x)
        edge_features_bond = self.bond_edge_embedding(bond_data.edge_attr)
        
        for i, (message_layer, layer_norm) in enumerate(zip(self.bond_message_layers, self.bond_layer_norms)):
            x_residual = x_bond
            x_bond = message_layer(x_bond, bond_data.edge_index, edge_features_bond)
            x_bond = layer_norm(x_bond)
            if i > 0:
                x_bond = x_bond + 0.3 * x_residual
            x_bond = F.silu(x_bond)
        
        attention_weights_bond = self.bond_attention(x_bond)
        x_bond_weighted = x_bond * attention_weights_bond
        
        bond_set2set = self.set2set_pool(x_bond_weighted, bond_data.batch)
        bond_mean = self.mean_pool(x_bond, bond_data.batch)
        bond_max = self.max_pool(x_bond, bond_data.batch)
        
        # ============ Atom Graph Forward (2-body) ============
        x_atom = self.atom_embedding(atom_data.x)
        edge_features_atom = self.atom_edge_embedding(atom_data.edge_attr)
        
        for i, (message_layer, layer_norm) in enumerate(zip(self.atom_message_layers, self.atom_layer_norms)):
            x_residual = x_atom
            x_atom = message_layer(x_atom, atom_data.edge_index, edge_features_atom)
            x_atom = layer_norm(x_atom)
            if i > 0:
                x_atom = x_atom + 0.3 * x_residual
            x_atom = F.silu(x_atom)
        
        attention_weights_atom = self.atom_attention(x_atom)
        x_atom_weighted = x_atom * attention_weights_atom
        
        atom_set2set = self.set2set_pool(x_atom_weighted, atom_data.batch)
        atom_mean = self.mean_pool(x_atom, atom_data.batch)
        atom_max = self.max_pool(x_atom, atom_data.batch)
        
        # ============ Torsion Graph Forward (4-body) ============
        x_torsion = self.torsion_embedding(torsion_data.x)
        edge_features_torsion = self.torsion_edge_embedding(torsion_data.edge_attr)
        
        for i, (message_layer, layer_norm) in enumerate(zip(self.torsion_message_layers, self.torsion_layer_norms)):
            x_residual = x_torsion
            x_torsion = message_layer(x_torsion, torsion_data.edge_index, edge_features_torsion)
            x_torsion = layer_norm(x_torsion)
            if i > 0:
                x_torsion = x_torsion + 0.3 * x_residual
            x_torsion = F.silu(x_torsion)
        
        attention_weights_torsion = self.torsion_attention(x_torsion)
        x_torsion_weighted = x_torsion * attention_weights_torsion
        
        torsion_set2set = self.set2set_pool(x_torsion_weighted, torsion_data.batch)
        torsion_mean = self.mean_pool(x_torsion, torsion_data.batch)
        torsion_max = self.max_pool(x_torsion, torsion_data.batch)
        
        # ============ Global Features ============
        global_features = self.global_mlp(bond_data.u)
        
        # ============ Combine & Predict ============
        combined = torch.cat([
            bond_set2set, bond_mean, bond_max,
            atom_set2set, atom_mean, atom_max,
            torsion_set2set, torsion_mean, torsion_max,
            global_features
        ], dim=1)
        
        out = self.final_mlp(combined)
        
        return out.squeeze()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = TripleGraphGNN(n_bond_features=6, n_atom_features=76, n_torsion_features=6, edge_dim=1).to(device)
print(f"✅ Triple Graph GNN Model: {sum(p.numel() for p in model.parameters()):,} params")
print(f"   Features:")
print(f"   - Bond graph (nodes=bonds, edges=angles) → 3-body")
print(f"   - Atom graph (nodes=atoms, edges=distances) → 2-body")
print(f"   - Torsion graph (nodes=planes, edges=torsions) → 4-body")
print(f"   - Separate message passing for each graph")
print(f"   - Combined pooling + MLP")


✅ Triple Graph GNN Model: 175,738 params
   Features:
   - Bond graph (nodes=bonds, edges=angles) → 3-body
   - Atom graph (nodes=atoms, edges=distances) → 2-body
   - Torsion graph (nodes=planes, edges=torsions) → 4-body
   - Separate message passing for each graph
   - Combined pooling + MLP


In [11]:
def train_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0
    for bond_data, atom_data, torsion_data in loader:
        bond_data = bond_data.to(device)
        atom_data = atom_data.to(device)
        torsion_data = torsion_data.to(device)
        optimizer.zero_grad()
        pred = model(bond_data, atom_data, torsion_data)
        loss = F.mse_loss(pred, bond_data.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * bond_data.num_graphs
    return total_loss / len(loader.dataset)

def evaluate(model, loader, device):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for bond_data, atom_data, torsion_data in loader:
            bond_data = bond_data.to(device)
            atom_data = atom_data.to(device)
            torsion_data = torsion_data.to(device)
            preds.extend(model(bond_data, atom_data, torsion_data).cpu().numpy())
            targets.extend(bond_data.y.cpu().numpy())
    return mean_absolute_error(targets, preds)

print("✅ Training functions ready")


✅ Training functions ready


In [12]:
from tqdm import tqdm

# Get fold 0 data
fold_num = 0
train_inputs, train_outputs = task.get_train_and_val_data(fold_num)
test_inputs, test_outputs = task.get_test_data(fold_num, include_target=True)

# Convert to graphs
print(f"Converting {len(train_inputs)} train structures to graphs...")
train_bond_graphs = []
train_atom_graphs = []
train_torsion_graphs = []
for i in tqdm(range(len(train_inputs)-1000)):
    bond_graph = structure_to_graph(train_inputs.iloc[i], train_outputs.iloc[i], df_ptable)
    atom_graph = structure_to_graph2(train_inputs.iloc[i], train_outputs.iloc[i], df_ptable)
    torsion_graph = structure_to_graph3(train_inputs.iloc[i], train_outputs.iloc[i], df_ptable)
    train_bond_graphs.append(bond_graph)
    train_atom_graphs.append(atom_graph)
    train_torsion_graphs.append(torsion_graph)

print(f"Converting {len(test_inputs)} test structures to graphs...")
test_bond_graphs = []
test_atom_graphs = []
test_torsion_graphs = []
for i in tqdm(range(len(test_inputs)-240)):
    bond_graph = structure_to_graph(test_inputs.iloc[i], test_outputs.iloc[i], df_ptable)
    atom_graph = structure_to_graph2(test_inputs.iloc[i], test_outputs.iloc[i], df_ptable)
    torsion_graph = structure_to_graph3(test_inputs.iloc[i], test_outputs.iloc[i], df_ptable)
    test_bond_graphs.append(bond_graph)
    test_atom_graphs.append(atom_graph)
    test_torsion_graphs.append(torsion_graph)


Converting 1012 train structures to graphs...


100%|██████████| 12/12 [00:01<00:00,  7.57it/s]


Converting 253 test structures to graphs...


100%|██████████| 13/13 [00:03<00:00,  3.86it/s]


In [13]:
# Create loaders
train_dataset = list(zip(train_bond_graphs, train_atom_graphs, train_torsion_graphs))
test_dataset = list(zip(test_bond_graphs, test_atom_graphs, test_torsion_graphs))

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Train model
model = TripleGraphGNN(n_bond_features=6, n_atom_features=76, n_torsion_features=6, edge_dim=1).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)

# Scheduler with better patience
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=0.01, epochs=1500, steps_per_epoch=len(train_loader),
    pct_start=0.3, anneal_strategy='cos', div_factor=25.0, final_div_factor=1e4
)

print(f"\n🚀 Training triple graph model...")
best_mae = float('inf')
patience = 0
max_patience = 1000

for epoch in range(1500):
    # Train
    model.train()
    total_loss = 0
    for bond_data, atom_data, torsion_data in train_loader:
        bond_data = bond_data.to(device)
        atom_data = atom_data.to(device)
        torsion_data = torsion_data.to(device)
        optimizer.zero_grad()
        
        pred = model(bond_data, atom_data, torsion_data)
        loss = F.mse_loss(pred, bond_data.y)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item() * bond_data.num_graphs
    
    train_loss = total_loss / len(train_dataset)
    
    # Evaluate every 5 epochs
    if epoch % 5 == 0 or epoch == 199:
        test_mae = evaluate(model, test_loader, device)
        current_lr = optimizer.param_groups[0]['lr']
        
        if test_mae < best_mae:
            best_mae = test_mae
            patience = 0
            torch.save(model.state_dict(), f'best_matbench_model_fold{fold_num}.pt')
            print(f"✓ Epoch {epoch:3d} | Train: {train_loss:.2f} | Test: {test_mae:.2f} ⭐ | LR: {current_lr:.6f}")
        else:
            patience += 5
            if epoch % 20 == 0:
                print(f"  Epoch {epoch:3d} | Train: {train_loss:.2f} | Test: {test_mae:.2f} | LR: {current_lr:.6f}")
        
        if patience >= max_patience:
            print(f"\n⚠️ Early stop at epoch {epoch}")
            break

print(f"\n✅ Best MAE: {best_mae:.2f} cm⁻¹")



🚀 Training triple graph model...
✓ Epoch   0 | Train: 209547.77 | Test: 570.15 ⭐ | LR: 0.000400
✓ Epoch   0 | Train: 209547.77 | Test: 570.15 ⭐ | LR: 0.000400
✓ Epoch   5 | Train: 208545.52 | Test: 569.04 ⭐ | LR: 0.000404
✓ Epoch   5 | Train: 208545.52 | Test: 569.04 ⭐ | LR: 0.000404
✓ Epoch  10 | Train: 208105.27 | Test: 568.71 ⭐ | LR: 0.000414
✓ Epoch  10 | Train: 208105.27 | Test: 568.71 ⭐ | LR: 0.000414
✓ Epoch  15 | Train: 207921.20 | Test: 568.52 ⭐ | LR: 0.000430
✓ Epoch  15 | Train: 207921.20 | Test: 568.52 ⭐ | LR: 0.000430
✓ Epoch  20 | Train: 207741.27 | Test: 568.39 ⭐ | LR: 0.000452
✓ Epoch  20 | Train: 207741.27 | Test: 568.39 ⭐ | LR: 0.000452
✓ Epoch  25 | Train: 207730.88 | Test: 568.26 ⭐ | LR: 0.000479
✓ Epoch  25 | Train: 207730.88 | Test: 568.26 ⭐ | LR: 0.000479


KeyboardInterrupt: 